# Cada hallazgo, una prueba
**8 minutos**

| Hallazgo | Prueba |
|---|---|
| Caso 1 y 3: un JOIN multiplica o pierde filas | El conteo de líneas no cambia tras los JOIN del dashboard |
| Caso 3: cliente duplicado | Llave única en `pedidos`, `clientes`, `productos` |
| Caso 3: productos que ya no existen | No hay líneas sin producto |
| Caso 3: corte de Valladolid tardío | Todas las cargas llegaron antes del corte |
| Todos | La diferencia entre reportes está dentro de una tolerancia acordada |

Las pruebas viven en `tests/test_calidad.py` y se ejecutan con `pytest`.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (funciona en Colab y en local)
import os, sys, subprocess, pathlib
REPO_URL = "https://github.com/jona866/taller-conciliacion-de-datos-datostada2026.git"  # ← el facilitador lo actualiza
if "google.colab" in sys.modules:
    destino = pathlib.Path("/content/numeros-no-cuadran")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
else:
    raiz = pathlib.Path.cwd()
    while not (raiz / "nnc.py").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    os.chdir(raiz)
sys.path.insert(0, os.getcwd())
import nnc
con = nnc.conectar("caso3")
print("Listo ✅  Tablas:", ", ".join(t for (t,) in con.execute("SHOW TABLES").fetchall()))

In [ ]:
print(open('tests/test_calidad.py', encoding='utf-8').read())

In [ ]:
import subprocess

def correr_pruebas(datos):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--tb=no", "-rf", "-p", "no:cacheprovider", "tests"],
                       env={**os.environ, "NNC_DATOS": datos}, capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-3000:])

correr_pruebas("caso3")

Fallan cinco: son exactamente lo que ustedes encontraron a mano en el caso 3.

## Llegó la carga de la primera semana de noviembre
Nadie ha abierto el dashboard todavía. Corramos las pruebas **antes**.

In [ ]:
correr_pruebas("carga_nueva")

In [ ]:
con_nueva = nnc.conectar("carga_nueva")
nnc.q(con_nueva, "SELECT * FROM productos WHERE producto_id IN (SELECT producto_id FROM productos GROUP BY 1 HAVING COUNT(*) > 1)")

Un producto se dio de alta otra vez con el mismo `producto_id` (“nuevo empaque”). El dashboard ya está contando doble esas ventas.
**Las pruebas lo dijeron antes que la Dirección.**

## Cierre
- Método de 7 pasos → `docs/checklist.md`
- El puente: si queda residuo, falta una causa; si suma cero, todavía hay que demostrar cada línea.
- **Trae tu caso:** describe a tu pareja una discrepancia real de tu trabajo (sin datos). ¿Por cuál de los 7 pasos empezarían?